# ViZDoom My Way Home: PPO + Curiosity (ICM) in PyTorch

In this notebook, we're going to train a PPO agent with curiosity to beat the My Way Home scenario in ViZDoom, using only visual observations from the game!

The other notebooks in this series lean on AgileRL. This one doesn't. Adding curiosity means touching the rollout, the reward, the networks, and the loss, which is most of PPO anyway, so everything here is written in plain PyTorch. Each piece is small and lives in its own cell: the encoder, the actor-critic, the curiosity module, the rollout buffer, and the PPO update. That makes it easy to swap one out and see what changes.

My Way Home is a maze. Doomguy is dropped into a random room, and a green armor vest is always in the same one. The rooms are different colors, so the screen is basically the map. The official writeup of the scenario is [here](https://vizdoom.farama.org/environments/default/#my-way-home). The catch is that the reward is very sparse: nothing happens until he stumbles onto the vest, so an agent that's just wandering gets almost no signal to learn from.

That makes it the perfect map for curiosity. It's the same map Pathak et al. used to introduce the Intrinsic Curiosity Module (ICM) in [Curiosity-driven Exploration by Self-supervised Prediction](https://arxiv.org/abs/1705.05363). The idea is to reward the agent for being surprised. A small forward model tries to predict the feature embedding of the next frame from the current frame's embedding and the action taken. Wherever that prediction is bad, Doomguy hasn't seen enough of that part of the maze, and the error gets paid out as a bonus reward. As places become familiar they stop paying, which pushes him on to somewhere new.

This one uses the stock maze. The episode ends when he reaches the vest, or when time runs out. In this version, Doomguy always walks forward and only chooses how to steer. During training, every step he spends running into a wall costs a penalty, the episode ends early with a penalty if he turns around twice in the same direction, and he earns a small bonus for every step he doesn't hit a wall. That's the steering and reward shaping sections below.

It also tries to use the original Doom WADs if it can find them.

The notebook was designed to run in place in the repo, however it also supports running on Google Colab, with the option of using Google Drive to save artifacts.

# Setup

Install the pinned dependencies first.

In [1]:
import sys

# Update the required system packages when running on Colab.
if "google.colab" in sys.modules:
    !apt-get update -qq
    !apt-get install -y -qq libopenal1 libopenal-dev > /dev/null

# Install the Python packages used throughout the notebook.
%pip install -q vizdoom==1.3.1 tqdm==4.67.3 gymnasium==1.3.0 imageio==2.38.0 imageio-ffmpeg==0.6.0 opencv-python==5.0.0.93 scipy==1.17.1 numpy==2.4.6 pandas==3.0.5 torch==2.13.0 matplotlib==3.11.1

Note: you may need to restart the kernel to use updated packages.


# Imports

In [2]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime
from functools import partial
from pathlib import Path

import gymnasium
import imageio.v2 as imageio
import imageio_ffmpeg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from gymnasium.spaces import Box, Discrete
from gymnasium.wrappers import FrameStackObservation, ResizeObservation, TransformObservation
from IPython.display import Image, display
from scipy.io import wavfile
from torch.distributions import Categorical
from tqdm.auto import tqdm


import vizdoom as vzd
from vizdoom import gymnasium_wrapper  # registers the Vizdoom* gymnasium envs
from vizdoom.gymnasium_wrapper.base_gymnasium_env import VizdoomEnv

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Experiment Setup
The observation setup matches the other notebooks. The agent acts every 4 frames and sees the last 4 frames stacked at 84x84, which gives it a sense of motion. My Way Home has 5 buttons, but the agent doesn't press them directly. It picks one of three steering actions, straight ahead, forward while turning left, or forward while turning right, which are explained in the steering section below. We run 16 environments side by side like the other PPO notebooks, because every update learns from a fresh batch collected across all of them.

Evaluation happens during training. At the end of every generation, the current policy plays `EVAL_EPISODES` episodes on a separate environment, and each one is saved as a `.lmp` demo while it plays. Those demos are tiny, so we keep all of them. After training, `RECORD_EPISODES` demos from the best generation are turned into videos. Set `RECORD_CHECKPOINTS` to `"all"` to convert that many from every generation too, although that takes a long while.

In [3]:
ENV_NAME = "VizdoomMyWayHome-v1"
NUM_ENVS = 16
FRAME_SKIP = 4          # act every 4 frames
STACK_SIZE = 4          # last 4 frames, so the agent can see motion
OBS_SIZE = (84, 84)
MAX_BUTTONS_PRESSED = 2 # forward and a turn together

EVAL_EPISODES = 5       # recorded evaluation episodes at the end of every generation
RECORD_EPISODES = 3     # demos per generation converted to video
RECORD_CHECKPOINTS = "best"  # or "all"

All the path logic lives in the next few cells, so the rest of the notebook doesn't need to care whether it's running on Colab or locally.

On Colab, `USE_DRIVE` keeps the run in Google Drive so it survives the session. Otherwise, everything stays in the notebook's folder.

In [4]:
USE_DRIVE = True # Colab only: keep the experiment (and every output) in Google Drive
EXPERIMENT = "my-way-home-ppo-icm-pytorch"

if "google.colab" in sys.modules and USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    experiment_dir = Path("/content/drive/MyDrive/rl-epsilon/vizdoom-agilerl-experiments") / EXPERIMENT
elif "google.colab" in sys.modules:
    experiment_dir = Path("/content") / EXPERIMENT  # lost when the Colab session ends
else:
    experiment_dir = Path.cwd()  # local: the notebook's own folder
print("experiment:", experiment_dir)

experiment: /Users/coldstart/Desktop/rl-epsilon/vizdoom-agilerl-experiments/my-way-home-ppo-icm-pytorch


ViZDoom ships with Freedoom. But if `DOOM2.WAD` is in the `WADS` folder, we use that instead. Always better to use the original assets in my opinion! The scenario is the stock maze, `my_way_home.cfg`, which ships with ViZDoom.

In [5]:
USE_DOOM2_WAD = True  # use DOOM2.WAD if found, otherwise ViZDoom's bundled Freedoom
STOCK_SCENARIO = "my_way_home.cfg"  # bundled with ViZDoom

scenario_cfg = str(Path(vzd.scenarios_path) / STOCK_SCENARIO)
SCENARIO_NAME = Path(scenario_cfg).stem

wad = experiment_dir.parent / "WADS" / "DOOM2.WAD"
DOOM_WAD = str(wad) if USE_DOOM2_WAD and wad.is_file() else None
IWAD_NAME = Path(DOOM_WAD).name if DOOM_WAD else "freedoom2"
print("scenario:", scenario_cfg)
print("IWAD:", IWAD_NAME)

scenario: /opt/miniconda3/lib/python3.13/site-packages/vizdoom/scenarios/my_way_home.cfg
IWAD: DOOM2.WAD


Each run gets its own timestamped folder. The best checkpoint, charts, and videos are copied to `artifacts`.

In [6]:
run_dir = experiment_dir / "training-runs" / f"run-{datetime.now():%Y%m%d_%H%M%S}"
checkpoint_dir = run_dir / "checkpoints"
recording_dir = run_dir / "recordings"
artifacts_dir = experiment_dir / "artifacts"
model_dir = artifacts_dir / "model"
video_dir = artifacts_dir / "video"
figure_dir = artifacts_dir / "plots_and_figures"
for d in (checkpoint_dir, recording_dir, model_dir, video_dir, figure_dir):
    d.mkdir(parents=True, exist_ok=True)

best_model_path = model_dir / "best_model.pt"
metrics_csv = figure_dir / "generation_metrics.csv"
results_path = artifacts_dir / "results.json"
print("run:", run_dir.name)

run: run-20261022_104146


# Environment

Next, we'll build the environment used to train Doomguy.

My Way Home is a maze of colored rooms. The scenario lets Doomguy turn, walk forward, or step sideways, though here he only gets the steering actions below. He starts in a random room, facing a random direction, and the green vest is always in the same room. Reaching it is worth +1. Every tic alive costs 0.0001, and the episode times out after 2100 tics.

`make_env` turns each frame into an 84x84 grayscale image and stacks the last 4 frames so the network can see motion. The curiosity bonus gets added later, inside the training loop, rather than in a wrapper, because it depends on the agent's own networks.

## Steering
`SteerActions` replaces the scenario's actions with three that all walk forward: straight ahead, forward while turning left, and forward while turning right. Each step is 4 tics. Forward is held for the first 2 and released for the last 2, while a turn is held for all 4. Pulsing forward like this caps his speed at about 15 units a step, half of what holding forward gives, so he has room to steer before he reaches a wall. The game env steps 2 tics at a time, and `SteerActions` plays the two halves of each step and adds up their rewards.

In [7]:
ACTION_NAMES = ("straight", "left", "right")


class SteerActions(gymnasium.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        base = env.unwrapped
        buttons = base.game.get_available_buttons()

        def combo(*held):
            # Index of the env action that presses exactly these buttons.
            target = [int(b in held) for b in buttons]
            return next(i for i, pressed in enumerate(base.button_map) if [int(p) for p in pressed] == target)

        forward, left, right = vzd.Button.MOVE_FORWARD, vzd.Button.TURN_LEFT, vzd.Button.TURN_RIGHT
        self.halves = [
            (combo(forward), combo()),
            (combo(forward, left), combo(left)),
            (combo(forward, right), combo(right)),
        ]
        self.action_space = Discrete(len(self.halves))

    def step(self, action):
        total = 0.0
        for half in self.halves[int(action)]:
            obs, reward, terminated, truncated, info = self.env.step(half)
            total += reward
            if terminated or truncated:
                break
        return obs, total, terminated, truncated, info

## Reward shaping
The earlier runs never learned the maze. First Doomguy spent most of each episode pressed against a wall or turning in place. Once walls ended the episode, he learned to stand nearly still and shimmy, turning and making tiny sidesteps, because any real movement risked a wall. So this run makes moving the default and punishes anything that slows exploration down. Hitting a wall no longer ends the episode, though. With forward locked in, crashes ended almost every early episode within a few steps, before he could see much of the maze. Now a crash just costs him, and he has to steer his way back out:

- **He always moves forward.** That's `SteerActions` above. He can't stand still, so the only way to survive is to steer.
- **Walking into a wall.** Every step, we check how far he got in the direction he was heading. In open floor, a step's own push always moves him the same distance, the first-step distance from a standstill that we measure below. Doomguy keeps his momentum, though, so the raw change in position mixes the step with the drift he already had. So we take out the drift first: Doom's friction keeps 90.6% of his speed each tic, so his current speed tells us how far he'd have drifted with no button pressed. What's left is the step's own movement. We project it onto his heading halfway through the step, so sliding along a wall counts as hitting it, not as moving. If that's less than `STUCK_FRACTION` (30%) of the standstill step, a wall or a corner stopped him, and that step costs `WALL_PENALTY`. The episode carries on, so grinding along a wall keeps costing him every step until he steers off it.
- **Turning around twice.** We add up how far he's turned since the episode started, with left turns positive and right turns negative. If the total reaches `SPIN_LIMIT` (720°, two full turns) either way, the episode ends with `SPIN_PENALTY`. Weaving left and right cancels out, so he can serpentine as much as he likes, but he can't circle.
- **Staying alive.** The game charges a little for every tic. In training, we refund that and pay `ALIVE_BONUS` for every step that doesn't hit a wall instead, so a longer clean run is always worth more.

`WALL_PENALTY` is 0.1 per step. Since a wall no longer ends the episode, a long grind would add up: a full episode against a wall at 1.0 a step would be worth about -500, which would swamp the value function. At 0.1, ten steps on a wall cost as much as the vest is worth. `SPIN_PENALTY` stays at 1.0, since it ends the episode. `ALIVE_BONUS` is kept small, at 0.001 a step, which comes to about 0.5 for a full episode. That's on purpose: reaching the vest ends the episode, so a large bonus for staying alive would teach him to avoid it. The shaping only applies to training. Evaluation and recording use the stock rewards, so the evaluation score is still the real game score.

In [8]:
SHAPING_HP = {
    "STUCK_FRACTION": 0.3,   # a step whose push moves him less than this share of its standstill step hit a wall
    "WALL_PENALTY": 0.1,     # penalty for every step that runs into a wall; the episode carries on
    "SPIN_LIMIT": 720,       # degrees of net turning in one direction that end the episode
    "SPIN_PENALTY": 1.0,     # penalty for reaching SPIN_LIMIT, which also ends the episode
    "ALIVE_BONUS": 0.001,    # reward per step for staying alive, in place of the game's time cost
}

`ShapingWrapper` applies the rules. It reads Doomguy's position, speed, and facing straight from the game. The drift is his speed times the friction carried over each of the `FRAME_SKIP` tics in a step. It marks every wall step with `info["wall_hit"]`, and when it ends an episode, it puts the reason in `info["ended_by"]`, so the training loop can count both.

`make_env` always adds `SteerActions`, since the policy only knows those three actions. It gets a `shaping` flag, off by default, so evaluation and recording stay on the stock rewards.

In [9]:
FRICTION = 0.90625  # Doom keeps this share of the player's speed each tic


def position(env):
    game = env.unwrapped.game
    return np.array([game.get_game_variable(vzd.GameVariable.POSITION_X),
                     game.get_game_variable(vzd.GameVariable.POSITION_Y)])


def velocity(env):
    game = env.unwrapped.game
    return np.array([game.get_game_variable(vzd.GameVariable.VELOCITY_X),
                     game.get_game_variable(vzd.GameVariable.VELOCITY_Y)])


def heading(env):
    return env.unwrapped.game.get_game_variable(vzd.GameVariable.ANGLE)


def turned(before, after):
    # Signed turn in degrees between two headings, positive to the left.
    return (after - before + 180) % 360 - 180


def unit(degrees):
    return np.array([np.cos(np.radians(degrees)), np.sin(np.radians(degrees))])


def forward_progress(start, start_heading, end, end_heading, drift=0.0):
    # How far a step's own push moved him along his heading halfway through the step.
    return np.dot(end - start - drift, unit(start_heading + turned(start_heading, end_heading) / 2))


class ShapingWrapper(gymnasium.Wrapper):
    def __init__(self, env, min_step):
        super().__init__(env)
        self.min_step = min_step  # {action: forward progress of its first step from a standstill}
        self.coast = sum(FRICTION**k for k in range(FRAME_SKIP))  # one step of drift per unit of speed
        self.time_cost = -env.unwrapped.game.get_living_reward() * FRAME_SKIP  # what the game charges per step

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.rotation = 0.0  # net degrees turned since the episode started, positive to the left
        return obs, info

    def step(self, action):
        action = int(action)
        start, start_heading = position(self.env), heading(self.env)
        drift = velocity(self.env) * self.coast
        obs, reward, terminated, truncated, info = self.env.step(action)
        if terminated or truncated:
            return obs, reward, terminated, truncated, info

        end_heading = heading(self.env)
        self.rotation += turned(start_heading, end_heading)
        progress = forward_progress(start, start_heading, position(self.env), end_heading, drift)

        wall_hit = progress < SHAPING_HP["STUCK_FRACTION"] * self.min_step[action]
        info = {**info, "wall_hit": wall_hit}
        if wall_hit:
            reward -= SHAPING_HP["WALL_PENALTY"]
        else:
            reward += self.time_cost + SHAPING_HP["ALIVE_BONUS"]
        if abs(self.rotation) >= SHAPING_HP["SPIN_LIMIT"]:
            reward -= SHAPING_HP["SPIN_PENALTY"]
            terminated = True
            info["ended_by"] = "spin"
        return obs, reward, terminated, truncated, info


def make_env(config_file=None, shaping=False):
    # Stock maze with the three steering actions. Pass a config file to load my_way_home.cfg directly, and shaping=True for training.
    kwargs = dict(
        render_mode="rgb_array",
        frame_skip=FRAME_SKIP // 2,  # SteerActions plays each step as two halves
        screen_resolution=vzd.ScreenResolution.RES_200X125,
        max_buttons_pressed=MAX_BUTTONS_PRESSED,
        treat_episode_timeout_as_truncation=False,
    )
    if DOOM_WAD:
        kwargs["doom_game_path"] = DOOM_WAD
    env = VizdoomEnv(config_file=config_file, **kwargs) if config_file else gymnasium.make(ENV_NAME, **kwargs)
    env = SteerActions(env)
    if shaping:
        env = ShapingWrapper(env, MIN_STEP)

    h, w, _ = env.observation_space["screen"].shape
    env = TransformObservation(
        env,
        lambda obs: (obs["screen"] @ np.array([0.299, 0.587, 0.114])).astype(np.uint8),
        Box(low=0, high=255, shape=(h, w), dtype=np.uint8),
    )  # grayscale
    env = ResizeObservation(env, OBS_SIZE)  # 84x84
    env = FrameStackObservation(env, stack_size=STACK_SIZE)  # last 4 frames
    return env

## Measuring the threshold
The wall rule needs one number per action from the game: how far a step moves him forward from a standstill. Rather than hardcode it, we measure it here with the same steps used in training. For each action, we reset into a few random starts, take one step, and keep the median, so one start facing a wall can't throw it off.

With forward pulsed for 2 of every 4 tics, the first step moves him about 4.8 units, so a step that gets him less than about 1.5 units of his own push forward counts as a wall. Turning settles at about 14° a step, so turning one way the whole time hits the 720° limit in about 51 steps.

In [10]:
def measure_movement(n_starts=5):
    env = make_env()
    min_step = {}
    for action in range(env.action_space.n):
        progress = []
        for seed in range(n_starts):
            env.reset(seed=seed)
            start, start_heading = position(env), heading(env)
            env.step(action)
            progress.append(forward_progress(start, start_heading, position(env), heading(env)))
        min_step[action] = float(np.median(progress))
    env.close()
    return min_step


MIN_STEP = measure_movement()
for action, distance in MIN_STEP.items():
    print(f"{ACTION_NAMES[action]}: first step {distance:.2f} units forward from a standstill, "
          f"wall below {SHAPING_HP['STUCK_FRACTION'] * distance:.2f}")

objc[82372]: Class SDLApplication is implemented in both /opt/miniconda3/lib/python3.13/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x14c5a4a68) and /opt/miniconda3/lib/python3.13/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x15d988890). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[82372]: Class SDLAppDelegate is implemented in both /opt/miniconda3/lib/python3.13/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x14c5a4ab8) and /opt/miniconda3/lib/python3.13/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x15d9888e0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[82372]: Class SDLTranslatorResponder is implemented in both /opt/miniconda3/lib/python3.13/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x14c5a4b30) and /opt/miniconda3/lib/python3.13/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x15d988958). This may cause spurio

straight: first step 4.84 units forward from a standstill, wall below 1.45
left: first step 4.84 units forward from a standstill, wall below 1.45
right: first step 4.84 units forward from a standstill, wall below 1.45


Training, evaluation, and recording all use the same stock maze, but only training gets the shaping. We create `NUM_ENVS` copies, each in its own process, so they can step at the same time.

In [11]:
env = gymnasium.vector.AsyncVectorEnv(
    [partial(make_env, shaping=True)] * NUM_ENVS, autoreset_mode=gymnasium.vector.AutoresetMode.SAME_STEP
)

objc[82385]: Class SDLApplication is implemented in both /opt/miniconda3/lib/python3.13/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x107804a68) and /opt/miniconda3/lib/python3.13/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x1162c4890). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[82385]: Class SDLAppDelegate is implemented in both /opt/miniconda3/lib/python3.13/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x107804ab8) and /opt/miniconda3/lib/python3.13/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x1162c48e0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[82385]: Class SDLTranslatorResponder is implemented in both /opt/miniconda3/lib/python3.13/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x107804b30) and /opt/miniconda3/lib/python3.13/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x1162c4958). This may cause spurio

# Hyperparameters
PPO collects `LEARN_STEP` steps of fresh experience, 2048 in total, or 128 from each of the 16 environments. It then makes `UPDATE_EPOCHS` passes over that batch in chunks of `BATCH_SIZE` and throws the batch away. `CLIP_COEF` limits how far one update can move the policy away from the policy that collected the data, and `ENT_COEF` adds a small bonus for keeping the action probabilities spread out. These are mostly the same starting values as the other PPO notebooks. `ENT_COEF` is the exception: it's 0.03 instead of 0.01, because in the last run the policy had settled on nearly fixed behaviour by generation 20, with entropy down around 0.15 against a maximum of 1.8 for its six actions. With three actions here, the maximum is 1.1.

We train for about 6,000,000 steps, triple the first run. Every 25 updates, 51,200 steps, make up one generation: we play and record the evaluation episodes and save a checkpoint, which gives us 120 to compare.

The curiosity settings are explained with the curiosity module below.

In [12]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"  # Apple Silicon GPU
else:
    device = "cpu"
print("device:", device)

PPO_HP = {
    "BATCH_SIZE": 256,
    "LR": 2.5e-4,
    "GAMMA": 0.99,             # how much future reward counts
    "GAE_LAMBDA": 0.95,        # how far ahead the advantage looks
    "LEARN_STEP": 2048,        # fresh steps across all envs before an update
    "UPDATE_EPOCHS": 4,        # passes over that batch
    "CLIP_COEF": 0.1,          # how far one update can move the policy
    "ENT_COEF": 0.03,          # bonus for keeping actions spread out
    "VF_COEF": 0.5,            # weight of the value loss
    "MAX_GRAD_NORM": 0.5,      # gradient clipping
    "MAX_STEPS": 6_144_000,
    "EVO_STEPS": 51_200,       # steps per generation, 25 updates
}

ICM_HP = {
    "INTRINSIC_COEF": 0.0,     # curiosity reward per step, after normalizing; 0 turns it off (0.01 for curiosity)
    "ICM_COEF": 0.0,           # weight of the ICM loss next to the PPO loss; 0 keeps ICM out of the encoder (1.0 for curiosity)
    "FORWARD_WEIGHT": 0.2,     # beta: forward loss vs inverse loss inside ICM
    "HIDDEN_SIZE": 512,        # hidden layer of the inverse and forward models
    "EPISODIC_CURIOSITY": True,   # True: dying gives up all future curiosity
}

NET_CONFIG = {
    "channels": (64, 128, 128),  # conv layers, double the Atari DQN widths
    "feature_dim": 512,          # encoder output, which is also ICM's feature space
    "hidden_size": 512,          # hidden layer of the policy and value heads
}
OBS_SHAPE = env.single_observation_space.shape
N_ACTIONS = int(env.single_action_space.n)

device: mps


# Networks
## Encoder
The encoder follows the shape of the original Atari DQN: three convolutional layers, then a linear layer down to a feature embedding. It's twice as wide, though, with 64, 128, and 128 filters and a 512-dimensional embedding, since the first run's encoder was the smaller Atari one and the maze has a lot of rooms to tell apart. That embedding is what both PPO heads and the curiosity module work from. It takes the raw uint8 frames and scales them to [0, 1] itself, so the rest of the notebook never has to remember to. The sizes all come from `NET_CONFIG`, and the size of the flattened conv output is measured with a dummy frame rather than hardcoded.

The embedding goes through a `LayerNorm` with no learnable scale before the final ReLU. In the last run, the features kept growing in magnitude, and the forward model's error grew right along with them, from about 40 to about 500. The curiosity bonus was measuring how big the features had gotten, not how unfamiliar a place was. With every embedding normalized to the same scale, the forward error can only stay high where the forward model really can't predict what happens next.

`layer_init` is the orthogonal initialization most PPO implementations use. It's one of those small details that makes a surprising difference to how quickly PPO gets going.

In [13]:
def layer_init(layer, gain=np.sqrt(2), bias=0.0):
    nn.init.orthogonal_(layer.weight, gain)
    nn.init.constant_(layer.bias, bias)
    return layer


class Encoder(nn.Module):
    def __init__(self, obs_shape, channels, feature_dim):
        super().__init__()
        c1, c2, c3 = channels
        self.conv = nn.Sequential(
            layer_init(nn.Conv2d(obs_shape[0], c1, kernel_size=8, stride=4)), nn.ReLU(),
            layer_init(nn.Conv2d(c1, c2, kernel_size=4, stride=2)), nn.ReLU(),
            layer_init(nn.Conv2d(c2, c3, kernel_size=3, stride=1)), nn.ReLU(),
            nn.Flatten(),
        )
        with torch.no_grad():
            conv_out = self.conv(torch.zeros(1, *obs_shape)).shape[1]
        self.linear = nn.Sequential(
            layer_init(nn.Linear(conv_out, feature_dim)),
            nn.LayerNorm(feature_dim, elementwise_affine=False),  # no learnable gain, so the scale stays fixed
            nn.ReLU(),
        )

    def forward(self, obs):
        return self.linear(self.conv(obs.float() / 255.0))

## Actor-critic
`ActorCritic` puts two heads on one shared encoder. The policy head turns the embedding into a probability for each action, and the value head estimates how well things are going so the actor can tell whether an action beat expectations.

The value head has two outputs: one for the game's reward and one for the curiosity reward. That's the setup from Burda et al.'s [Exploration by Random Network Distillation](https://arxiv.org/abs/1810.12894). Keeping them apart lets each reward be discounted under its own rules, which is what makes non-episodic curiosity possible (see the rollout buffer below). Column 0 is extrinsic and column 1 is intrinsic.

The encoder and the heads are exposed separately. `features` runs only the encoder, and `heads` runs only the heads on an embedding we already have, so a single encoder pass serves PPO and ICM. `act` is the convenience wrapper for stepping the environment.

The policy's output layer starts with tiny weights so every action starts out about equally likely, and the value head starts at unit scale. Those gains also come from the standard PPO recipe.

In [14]:
class ActorCritic(nn.Module):
    def __init__(self, obs_shape, n_actions, channels, feature_dim, hidden_size):
        super().__init__()
        self.encoder = Encoder(obs_shape, channels, feature_dim)
        self.policy_head = nn.Sequential(
            layer_init(nn.Linear(feature_dim, hidden_size)), nn.ReLU(),
            layer_init(nn.Linear(hidden_size, n_actions), gain=0.01),
        )
        self.value_head = nn.Sequential(
            layer_init(nn.Linear(feature_dim, hidden_size)), nn.ReLU(),
            layer_init(nn.Linear(hidden_size, 2), gain=1.0),  # (extrinsic, intrinsic)
        )

    def features(self, obs):
        return self.encoder(obs)

    def heads(self, features):
        # Returns (action distribution, values) for a batch of embeddings; values are (batch, 2).
        return Categorical(logits=self.policy_head(features)), self.value_head(features)

    @torch.no_grad()
    def act(self, obs):
        # Sample actions for a batch of raw observations; returns (actions, log_probs, values).
        dist, values = self.heads(self.features(obs))
        actions = dist.sample()
        return actions, dist.log_prob(actions), values

## Intrinsic Curiosity Module
ICM sits on top of the feature embedding and has two small networks.

- The **inverse model** looks at the embeddings of two consecutive frames and guesses which action took Doomguy from one to the other. Training it shapes the encoder to keep what the agent can affect, like walls moving past as he turns, and to ignore what it can't.
- The **forward model** takes the current embedding and the action, and predicts the next embedding. Its error is the curiosity reward.

The ICM loss mixes the two as \((1-\beta)\,L_I + \beta\,L_F\), with \(\beta = 0.2\) as in the paper. The forward model's inputs and target are detached from the encoder. Because the encoder is shared with the policy, letting the forward loss reach it would let the encoder make prediction easy by squashing the features together, which would also starve the policy. This way the encoder is shaped by the inverse loss and by PPO, and the forward model just learns on top of it.

In [15]:
class ICM(nn.Module):
    def __init__(self, feature_dim, n_actions, hidden_size=256):
        super().__init__()
        self.n_actions = n_actions
        self.inverse_model = nn.Sequential(
            nn.Linear(2 * feature_dim, hidden_size), nn.ReLU(), nn.Linear(hidden_size, n_actions),
        )
        self.forward_model = nn.Sequential(
            nn.Linear(feature_dim + n_actions, hidden_size), nn.ReLU(), nn.Linear(hidden_size, feature_dim),
        )

    def forward(self, features, next_features, actions):
        # Returns (forward_error, inverse_logits); forward_error is per sample.
        inverse_logits = self.inverse_model(torch.cat([features, next_features], dim=-1))
        one_hot = F.one_hot(actions, self.n_actions).float()
        predicted = self.forward_model(torch.cat([features.detach(), one_hot], dim=-1))
        forward_error = 0.5 * (predicted - next_features.detach()).pow(2).sum(dim=-1)
        return forward_error, inverse_logits

`INTRINSIC_COEF` sets how much curiosity is worth next to the vest. The raw forward error has no natural scale. It starts out huge while the forward model is untrained, collapses within an update or two, then drifts as the encoder changes. So `curiosity_rewards` divides each rollout's errors by that rollout's mean error before scaling. That makes the average bonus exactly `INTRINSIC_COEF` per step. Steps that surprised the forward model more than usual earn more, and steps it predicted well earn less. At 0.01, a hundred steps of fresh territory is worth about one vest. Set it to 0 to get a plain PPO baseline with the exact same code.

An earlier version divided by a running standard deviation of every error seen so far, which is closer to what Burda et al. do. With the LayerNorm'd features, though, the errors in a batch sit tightly around their mean, so error divided by spread came out around 50 times too big on the first update. The one wild first batch then inflated the running statistics, starving the agent of curiosity for the next several updates. Dividing by the batch mean is immune to both problems. Because curiosity is episodic in this run, that constant average level also works as a reward for surviving, on top of `ALIVE_BONUS`.

`ICM_COEF` weights the ICM loss next to the PPO loss when they're added together. The paper scales the policy loss down by 0.1 instead, which works out to an ICM weight of about 10. We start at 1 because the encoder here is shared with the policy.

In [16]:
def curiosity_rewards(errors):
    return ICM_HP["INTRINSIC_COEF"] * errors / (errors.mean() + 1e-8)

# PPO
## Rollout buffer
`RolloutBuffer` holds one batch of experience as arrays shaped `(steps, envs, ...)`. Observations stay as uint8 on the CPU, about 115 MB for a full batch, and only move to the GPU a minibatch at a time.

It stores `next_obs` as well, because ICM needs the frame each action actually led to.

`gae` is standard generalized advantage estimation. `dones[t]` means step `t` ended the episode, so the bootstrap from the next value is cut right there. `compute_advantages` runs it twice, once per value head, and adds the two advantages together for the policy update.

The game reward is episodic: reaching the vest or timing out ends the episode, and nothing after it counts. In this run, curiosity is episodic too, so ending the episode by spinning gives up every future curiosity bonus. That's on purpose. In an earlier run, curiosity was non-episodic: its GAE ignored episode ends, so the stream carried on into the next episode, and crashing into a wall, which ended the episode then, cost nothing but the penalty. Burda et al. found non-episodic curiosity explores better, because the agent never has to weigh ending the episode against future bonuses. With early endings in the mix, though, we want them to mean something. The catch is that reaching the vest ends the episode too, so it also costs future curiosity. Set `EPISODIC_CURIOSITY` to `False` to carry curiosity across episode ends again.

In [17]:
def gae(rewards, values, last_value, dones, gamma, gae_lambda):
    advantages = np.zeros_like(values)
    running = np.zeros_like(last_value)
    for t in reversed(range(len(rewards))):
        next_value = last_value if t == len(rewards) - 1 else values[t + 1]
        not_done = 1.0 - dones[t]
        delta = rewards[t] + gamma * next_value * not_done - values[t]
        running = delta + gamma * gae_lambda * not_done * running
        advantages[t] = running
    return advantages


class RolloutBuffer:
    def __init__(self, n_steps, num_envs, obs_shape):
        shape = (n_steps, num_envs)
        self.obs = np.zeros((*shape, *obs_shape), dtype=np.uint8)
        self.next_obs = np.zeros((*shape, *obs_shape), dtype=np.uint8)
        self.actions = np.zeros(shape, dtype=np.int64)
        self.log_probs = np.zeros(shape, dtype=np.float32)
        self.values = np.zeros((*shape, 2), dtype=np.float32)  # (extrinsic, intrinsic)
        self.rewards = np.zeros(shape, dtype=np.float32)
        self.dones = np.zeros(shape, dtype=np.float32)
        self.n_steps = n_steps

    def add(self, t, obs, next_obs, actions, log_probs, values, rewards, dones):
        self.obs[t], self.next_obs[t], self.actions[t] = obs, next_obs, actions
        self.log_probs[t], self.values[t] = log_probs, values
        self.rewards[t], self.dones[t] = rewards, dones

    def compute_advantages(self, intrinsic_rewards, last_values, gamma, gae_lambda, episodic_curiosity):
        intrinsic_dones = self.dones if episodic_curiosity else np.zeros_like(self.dones)
        extrinsic = gae(self.rewards, self.values[..., 0], last_values[:, 0], self.dones, gamma, gae_lambda)
        intrinsic = gae(intrinsic_rewards, self.values[..., 1], last_values[:, 1], intrinsic_dones, gamma, gae_lambda)
        self.advantages = extrinsic + intrinsic
        self.returns = np.stack([extrinsic, intrinsic], axis=-1) + self.values

    def minibatches(self, batch_size):
        # Yields shuffled minibatches as dicts of tensors on `device`, covering the buffer once.
        flat = lambda x: x.reshape(-1, *x.shape[2:])
        data = {k: flat(getattr(self, k)) for k in
                ("obs", "next_obs", "actions", "log_probs", "values", "advantages", "returns")}
        n = len(data["actions"])
        for idx in np.array_split(np.random.permutation(n), n // batch_size):
            yield {k: torch.as_tensor(v[idx], device=device) for k, v in data.items()}

## Collecting a rollout
`collect_rollout` steps all 16 environments `LEARN_STEP / NUM_ENVS` times and fills the buffer. It returns both values of the observation the rollout stopped on, which GAE uses to bootstrap, along with the score and the reason each finished episode ended: the vest, a timeout, or one of the shaping rules.

The vector env resets a finished environment in the same step, so `next_obs` already shows the new episode for that slot. The real last frame is in `info["final_obs"]`, and we swap it in. Otherwise the forward model would get paid for failing to predict a teleport to a random room.

In [18]:
def to_tensor(obs):
    return torch.as_tensor(obs, device=device)


def wall_hits(info):
    # Wall steps across envs; steps that ended an episode report their info in final_info.
    hits = 0
    for source in (info, info.get("final_info", {})):
        if "wall_hit" in source:
            hits += int(np.sum(source["wall_hit"].astype(bool) & source["_wall_hit"]))
    return hits


def collect_rollout(model, env, buffer, obs, scores):
    finished_scores, end_reasons, hits = [], [], 0
    for t in range(buffer.n_steps):
        actions, log_probs, values = model.act(to_tensor(obs))
        actions = actions.cpu().numpy()
        next_obs, reward, terminated, truncated, info = env.step(actions)
        done = terminated | truncated

        real_next_obs = next_obs.copy()
        for i in np.flatnonzero(info.get("_final_obs", [])):
            real_next_obs[i] = info["final_obs"][i]
        buffer.add(t, obs, real_next_obs, actions, log_probs.cpu().numpy(), values.cpu().numpy(), reward, done)

        scores += reward
        hits += wall_hits(info)
        ended_by = info.get("final_info", {}).get("ended_by", [None] * len(done))
        for i in np.flatnonzero(done):
            finished_scores.append(scores[i])
            end_reasons.append(ended_by[i] or ("vest" if scores[i] > 0 else "timeout"))
            scores[i] = 0.0
        obs = next_obs

    last_values = model.act(to_tensor(obs))[2].cpu().numpy()
    return obs, last_values, finished_scores, end_reasons, hits

## Curiosity reward
`forward_errors` runs the whole rollout back through the encoder and forward model to get the prediction error at every step. It's computed once per rollout with the current networks, so every transition in the batch is scored by the same model.

In [19]:
@torch.no_grad()
def forward_errors(model, icm, buffer, chunk=1024):
    obs = buffer.obs.reshape(-1, *buffer.obs.shape[2:])
    next_obs = buffer.next_obs.reshape(obs.shape)
    actions = to_tensor(buffer.actions.reshape(-1))
    errors = []
    for i in range(0, len(obs), chunk):
        features = model.features(to_tensor(obs[i:i + chunk]))
        next_features = model.features(to_tensor(next_obs[i:i + chunk]))
        error, _ = icm(features, next_features, actions[i:i + chunk])
        errors.append(error.cpu().numpy())
    return np.concatenate(errors).reshape(buffer.actions.shape)

## The update
`ppo_update` is PPO's clipped update with the ICM loss added on. Each minibatch embeds the current and next frames in one encoder pass. The current embedding feeds both PPO heads and ICM, and the next embedding feeds only ICM. Everything is summed into one loss and stepped together, so the inverse loss trains the shared encoder alongside the policy.

The pieces of the PPO loss:

- **Policy loss**: the probability ratio between the new and old policy, times the advantage, clipped to `1 ± CLIP_COEF` so one update can't move the policy too far.
- **Value loss**: squared error against the GAE returns, with the same clipping applied to how far the value can move. It's summed over both value heads.
- **Entropy**: subtracted, so spreading the probabilities out lowers the loss.

Advantages are normalized per minibatch. We also log how often the inverse model guesses the action right. Chance is 1 in 6, and if it never climbs above that, the encoder isn't learning anything about movement.

In [20]:
def ppo_update(model, icm, optimizer, buffer):
    clip, beta = PPO_HP["CLIP_COEF"], ICM_HP["FORWARD_WEIGHT"]
    params = [p for group in optimizer.param_groups for p in group["params"]]
    totals, n_minibatches = {}, 0
    for _ in range(PPO_HP["UPDATE_EPOCHS"]):
        for mb in buffer.minibatches(PPO_HP["BATCH_SIZE"]):
            features, next_features = model.features(torch.cat([mb["obs"], mb["next_obs"]])).chunk(2)
            dist, values = model.heads(features)
            log_probs = dist.log_prob(mb["actions"])
            entropy = dist.entropy().mean()

            advantages = (mb["advantages"] - mb["advantages"].mean()) / (mb["advantages"].std() + 1e-8)
            ratio = torch.exp(log_probs - mb["log_probs"])
            policy_loss = torch.max(-advantages * ratio, -advantages * torch.clamp(ratio, 1 - clip, 1 + clip)).mean()

            clipped_values = mb["values"] + torch.clamp(values - mb["values"], -clip, clip)
            value_loss = 0.5 * torch.max((values - mb["returns"]) ** 2, (clipped_values - mb["returns"]) ** 2).sum(-1).mean()

            forward_error, inverse_logits = icm(features, next_features, mb["actions"])
            forward_loss = forward_error.mean()
            inverse_loss = F.cross_entropy(inverse_logits, mb["actions"])
            icm_loss = (1 - beta) * inverse_loss + beta * forward_loss

            loss = (
                policy_loss
                + PPO_HP["VF_COEF"] * value_loss
                - PPO_HP["ENT_COEF"] * entropy
                + ICM_HP["ICM_COEF"] * icm_loss
            )
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(params, PPO_HP["MAX_GRAD_NORM"])
            optimizer.step()

            step_metrics = {
                "loss": loss, "policy_loss": policy_loss, "value_loss": value_loss, "entropy": entropy,
                "inverse_loss": inverse_loss, "forward_loss": forward_loss,
                "inverse_accuracy": (inverse_logits.argmax(-1) == mb["actions"]).float().mean(),
            }
            for k, v in step_metrics.items():
                totals[k] = totals.get(k, 0.0) + v.item()
            n_minibatches += 1
    return {k: v / n_minibatches for k, v in totals.items()}

## Evaluation and recording
Evaluation happens at the end of every generation, on its own environment so the 16 training environments never get interrupted mid-episode. The policy plays `EVAL_EPISODES` episodes with no curiosity bonus, so the score is the real game reward. Like the other PPO notebooks, it keeps sampling from the policy instead of always taking the most likely action, which can leave him stuck turning in place.

Every one of those episodes is recorded as a ViZDoom `.lmp` demo while it plays. A demo stores only the inputs, a few KB per episode, and gets replayed at full resolution later. The Gymnasium wrapper doesn't expose demo recording, so `reset_with_lmp` temporarily replaces the reset function with one that starts a recording. `flush_lmp` begins the next episode so the file is actually written.

One gotcha: ViZDoom hangs if the demo path is longer than about 112 characters, and the run folder on Colab with Drive is longer than that. So each demo is written under a short temporary name in the working directory and moved into the run's `recordings` folder once the generation's episodes are done.

In [21]:
def reset_with_lmp(env, lmp_path):
    base = env.unwrapped
    orig_reset = base.reset

    def recorded_reset(*, seed=None, options=None):
        gymnasium.Env.reset(base, seed=seed)
        game_seed = int(
            base.np_random.integers(0, np.iinfo(np.uint32).max + 1, dtype=np.uint32)
        )
        base.game.set_seed(game_seed)
        if not base.game.is_running():
            base.game.init()
        # ViZDoom hangs if the demo path is longer than ~112 characters, so pass it relative to the working directory
        base.game.new_episode(os.path.relpath(lmp_path))
        base.state = base.game.get_state()
        return base._VizdoomEnv__collect_observations(), {}

    base.reset = recorded_reset
    try:
        return env.reset()
    finally:
        base.reset = orig_reset


def flush_lmp(env):
    env.unwrapped.game.new_episode()

In [22]:
def play(env, model, obs):
    # Run one episode from `obs`; returns (total_reward, actions).
    total, actions, done = 0.0, [], False
    while not done:
        action = int(model.act(to_tensor(obs[None]))[0].item())
        obs, reward, terminated, truncated, _ = env.step(action)
        total += reward
        actions.append(action)
        done = terminated or truncated
    return total, actions


def evaluate_and_record(model, env, generation):
    # Play EVAL_EPISODES recorded episodes; returns (rewards, demo paths in recording_dir).
    rewards, temp_demos = [], []
    for episode in range(EVAL_EPISODES):
        temp_demo = Path(f"_eval_demo_{episode}.lmp")  # short and relative, for the path length limit
        obs, _ = reset_with_lmp(env, temp_demo)
        reward, _ = play(env, model, obs)
        rewards.append(reward)
        temp_demos.append(temp_demo)
    flush_lmp(env)

    demos = []
    for episode, temp_demo in enumerate(temp_demos):
        demo = recording_dir / f"gen_{generation:03d}_episode_{episode}.lmp"
        shutil.move(temp_demo, demo)
        demos.append(demo)
    return rewards, demos

A checkpoint is a single file holding the actor-critic, ICM, the optimizer, and the settings needed to rebuild the networks. `load_model` only rebuilds the actor-critic, since that's all it takes to play.

In [23]:
def save_checkpoint(path, model, icm, optimizer, steps):
    torch.save({
        "steps": steps,
        "config": {"obs_shape": OBS_SHAPE, "n_actions": N_ACTIONS, **NET_CONFIG},
        "model": model.state_dict(),
        "icm": icm.state_dict(),
        "optimizer": optimizer.state_dict(),
        "ppo_hp": PPO_HP,
        "icm_hp": ICM_HP,
    }, path)


def load_model(path):
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    model = ActorCritic(**checkpoint["config"]).to(device)
    model.load_state_dict(checkpoint["model"])
    return model.eval()

# Training
Now we build everything and run it. One optimizer covers the actor-critic and ICM. The `eps` of 1e-5 is another PPO convention.

In [24]:
model = ActorCritic(OBS_SHAPE, N_ACTIONS, **NET_CONFIG).to(device)
icm = ICM(NET_CONFIG["feature_dim"], N_ACTIONS, ICM_HP["HIDDEN_SIZE"]).to(device)
optimizer = torch.optim.Adam([*model.parameters(), *icm.parameters()], lr=PPO_HP["LR"], eps=1e-5)
buffer = RolloutBuffer(PPO_HP["LEARN_STEP"] // NUM_ENVS, NUM_ENVS, OBS_SHAPE)
eval_env = make_env(scenario_cfg)
print(f"actor-critic: {sum(p.numel() for p in model.parameters()):,} params, ICM: {sum(p.numel() for p in icm.parameters()):,} params")

actor-critic: 4,034,885 params, ICM: 1,053,187 params


Each update has four steps: collect a rollout, score it with curiosity, compute advantages for the game reward and the curiosity reward, and run the PPO and ICM losses together. Each generation runs 25 updates, then plays and records its evaluation episodes and saves a checkpoint.

Each generation logs one row: the PPO and ICM losses, the inverse model's accuracy, the spread of the curiosity reward, the mean score of the training episodes that finished (shaping penalties included), how those episodes ended, and the mean and standard deviation of the evaluation episodes. The evaluation rewards and demo paths for every generation are kept in `eval_rewards` and `eval_demos`.

At triple the steps and a wider network, this one is a long run. A good time to step away from the machine.

In [25]:
history, eval_rewards, eval_demos = [], [], []
updates_per_generation = PPO_HP["EVO_STEPS"] // PPO_HP["LEARN_STEP"]
total_steps = 0

obs, _ = env.reset()
scores = np.zeros(NUM_ENVS)
pbar = tqdm(total=PPO_HP["MAX_STEPS"], unit="step")
for generation in range(PPO_HP["MAX_STEPS"] // PPO_HP["EVO_STEPS"]):
    model.train()
    update_metrics, episode_scores, end_reasons, curiosity_spread, hits = [], [], [], [], 0
    for _ in range(updates_per_generation):
        obs, last_values, finished, reasons, rollout_hits = collect_rollout(model, env, buffer, obs, scores)
        episode_scores += finished
        end_reasons += reasons
        hits += rollout_hits

        errors = forward_errors(model, icm, buffer)
        intrinsic_rewards = curiosity_rewards(errors)
        curiosity_spread.append(errors.std() / errors.mean())

        buffer.compute_advantages(intrinsic_rewards, last_values, PPO_HP["GAMMA"], PPO_HP["GAE_LAMBDA"], ICM_HP["EPISODIC_CURIOSITY"])
        update_metrics.append(ppo_update(model, icm, optimizer, buffer))
        total_steps += PPO_HP["LEARN_STEP"]
        pbar.update(PPO_HP["LEARN_STEP"])

    model.eval()
    rewards, demos = evaluate_and_record(model, eval_env, generation)
    eval_rewards.append(rewards)
    eval_demos.append(demos)

    row = {"steps": total_steps, **pd.DataFrame(update_metrics).mean().to_dict()}
    row["curiosity_spread"] = float(np.mean(curiosity_spread))
    row["train_score"] = float(np.mean(episode_scores)) if episode_scores else np.nan
    for reason in ("vest", "timeout", "spin"):
        row[f"ended_{reason}"] = end_reasons.count(reason) / max(len(end_reasons), 1)
    row["wall_hit_rate"] = hits / (updates_per_generation * PPO_HP["LEARN_STEP"])
    row["eval_mean"], row["eval_std"] = float(np.mean(rewards)), float(np.std(rewards))
    history.append(row)
    pbar.set_postfix(eval=f"{row['eval_mean']:.2f}", vest=f"{row['ended_vest']:.0%}", wall=f"{row['wall_hit_rate']:.0%}", spin=f"{row['ended_spin']:.0%}")

    save_checkpoint(checkpoint_dir / f"ckpt_{total_steps}.pt", model, icm, optimizer, total_steps)
pbar.close()
env.close()
eval_env.close()

 63%|██████▎   | 3858432/6144000 [3:51:29<4:13:46, 150.10step/s, eval=-0.21, spin=0%, vest=98%, wall=3%]

KeyboardInterrupt: 

 63%|██████▎   | 3858432/6144000 [3:51:41<4:13:46, 150.10step/s, eval=-0.21, spin=0%, vest=98%, wall=3%]

# Best Generation
Every generation was already scored during training, so picking the best one is just a lookup. The generation with the highest mean evaluation reward wins; ties go to the later generation. Its checkpoint is copied to `artifacts`.

In [ ]:
best_gen = max(range(len(history)), key=lambda g: (history[g]["eval_mean"], g))
best_checkpoint = checkpoint_dir / f"ckpt_{history[best_gen]['steps']}.pt"
shutil.copy2(best_checkpoint, best_model_path)
print(f"best: gen {best_gen} ({best_checkpoint.name}), mean eval reward {history[best_gen]['eval_mean']:.2f}, "
      f"rewards={np.round(eval_rewards[best_gen], 2).tolist()}")

# Analysis
Let's see how it did! Training is noisy and the eval episodes aren't seeded, so your curves won't match these exactly. The notes below are about the general shape.

## Metrics
One row per generation: the PPO losses and entropy, the ICM losses and inverse accuracy, the curiosity spread, the training score, how training episodes ended, and the evaluation mean and standard deviation. The full table gets saved as a CSV in `artifacts`. Here we just show the first few generations, the best one, and the last few.

In [ ]:
metrics = pd.DataFrame(history)
metrics.index.name = "generation"
shown = pd.concat([metrics.head(3), metrics.loc[[best_gen]], metrics.tail(3)])
shown[~shown.index.duplicated()]

In [ ]:
metrics.to_csv(metrics_csv)

## Charts
Saved to `artifacts/plots_and_figures`, and shown here too.

In [ ]:
def save(fig, name):
    fig.savefig(figure_dir / name, dpi=150, bbox_inches="tight")
    plt.show()


gens = metrics.index

**Loss.** PPO's losses don't say much about how well the agent plays. The value loss is how far off the critic is, and with curiosity in the reward the critic is chasing a moving target, so expect it to stay noisy. The policy loss stays near zero by design, since clipping keeps every update small.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(gens, metrics["value_loss"], marker="o")
ax1.set(xlabel="Generation", ylabel="Mean value loss", title="Critic (value) loss per generation")
ax2.plot(gens, metrics["policy_loss"], marker="o")
ax2.set(xlabel="Generation", ylabel="Mean policy loss", title="Actor (policy) loss per generation")
save(fig, "loss.png")

**ICM.** The inverse accuracy is the health check for the feature embedding. It should climb well above chance, the dashed line, as the encoder learns what Doomguy's actions do to the view. The forward loss is how surprised the forward model is on average. It usually drops quickly, then levels off or creeps back up as he reaches parts of the maze it hasn't seen.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(gens, metrics["inverse_accuracy"], marker="o")
ax1.axhline(1 / N_ACTIONS, color="gray", linestyle="--", label="Chance")
ax1.set(xlabel="Generation", ylabel="Accuracy", title="Inverse model accuracy per generation")
ax1.legend()
ax2.plot(gens, metrics["forward_loss"], marker="o")
ax2.set(xlabel="Generation", ylabel="Mean forward loss", title="Forward model loss per generation")
save(fig, "icm_loss.png")

**Curiosity.** The average curiosity bonus is pinned at `INTRINSIC_COEF` per step by construction, so the useful number is how much it varies. This chart is the spread of the forward error within each rollout, its standard deviation divided by its mean. A high value means some places surprise the forward model far more than others, which is the signal pulling him toward new territory. Near zero, every step pays about the same and curiosity has stopped steering anything.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gens, metrics["curiosity_spread"], marker="o")
ax.set(xlabel="Generation", ylabel="Std / mean of forward error", title="Curiosity spread per generation")
save(fig, "curiosity_spread.png")

**Entropy.** Entropy measures how spread out the action probabilities are. It starts near the maximum, the dashed line where every action is equally likely, and should fall as the agent gets more confident. If it crashes toward zero early, the agent stopped exploring too soon.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gens, metrics["entropy"], marker="o")
ax.axhline(np.log(N_ACTIONS), color="gray", linestyle="--", label="Maximum (uniform policy)")
ax.set(xlabel="Generation", ylabel="Mean entropy", title="Policy entropy per generation")
ax.legend()
save(fig, "entropy.png")

**Average reward.** A found vest lands near 1. A timeout lands near -0.2, because every tic costs a tiny bit and the episode is long. Blue shows the recorded evaluation episodes at the end of each generation, with the shaded band representing one standard deviation. Orange shows the training episodes that finished during the generation. Those include the shaping, so every wall step subtracts `WALL_PENALTY`, a spin subtracts `SPIN_PENALTY`, and every clean step adds `ALIVE_BONUS` instead of the game's time cost.

The band stays wide while he's still wandering, since one vest and one timeout pull the average in opposite directions. If blue sits near 1, he found the vest.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
(line,) = ax.plot(gens, metrics["eval_mean"], marker="o", label=f"Evaluation ({EVAL_EPISODES} episodes, {SCENARIO_NAME})")
ax.fill_between(gens, metrics["eval_mean"] - metrics["eval_std"],
                metrics["eval_mean"] + metrics["eval_std"], color=line.get_color(), alpha=0.2)
ax.plot(gens, metrics["train_score"], marker="o", label="Training episodes (with shaping penalties)")
ax.axvline(best_gen, color="gray", linestyle="--", label=f"Best generation ({best_gen})")
ax.set(xlabel="Generation", ylabel="Average reward", title="Average reward per generation")
ax.legend()
save(fig, "average_reward.png")

**How training episodes ended, and how often he hit walls.** The left chart is the share of each generation's training episodes that ended at the vest, timed out, or turned around twice. Walls don't end episodes anymore, so the right chart shows the share of all training steps that ran into a wall. Watch whether the wall rate and spins shrink and the vest share grows. A timeout just means he kept moving until time ran out, which is fine as long as the vest share follows.

In [ ]:
fig, (ax, wall_ax) = plt.subplots(1, 2, figsize=(12, 4))
for reason in ("vest", "timeout", "spin"):
    ax.plot(gens, metrics[f"ended_{reason}"], marker="o", label=reason)
ax.axvline(best_gen, color="gray", linestyle="--", label=f"Best generation ({best_gen})")
ax.set(xlabel="Generation", ylabel="Share of training episodes", title="How training episodes ended")
ax.legend()
wall_ax.plot(gens, metrics["wall_hit_rate"], marker="o", color="tab:red")
wall_ax.axvline(best_gen, color="gray", linestyle="--")
wall_ax.set(xlabel="Generation", ylabel="Share of training steps", title="Steps that hit a wall")
save(fig, "episode_endings.png")

# Recording
The demos were all recorded during training, so all that's left is turning them into videos.

The first `RECORD_EPISODES` demos from the best generation are copied into `artifacts/video` as `best_episode_*.lmp`. With `RECORD_CHECKPOINTS` set to `"all"`, the same number from every other generation are converted in place in the run folder. Existing best-model videos are cleared first so they don't get mixed with the current run.

In [ ]:
assert RECORD_CHECKPOINTS in ("best", "all")
for stale in video_dir.glob("best_episode_*"):
    stale.unlink()

lmps, lmp_rewards = [], {}
for gen in range(len(history)) if RECORD_CHECKPOINTS == "all" else [best_gen]:
    for episode, demo in enumerate(eval_demos[gen][:RECORD_EPISODES]):
        if gen == best_gen:
            demo = Path(shutil.copy2(demo, video_dir / f"best_episode_{episode}.lmp"))
        lmps.append(demo)
        lmp_rewards[demo] = eval_rewards[gen][episode]
        print(f"gen {gen:>3}  episode {episode}  reward={eval_rewards[gen][episode]:.2f}  -> {demo.name}")

`lmp_to_mp4` replays each demo at 720p with the HUD enabled. The result looks much better than the 84x84 input used for training because we're replaying the actions in a fresh game rather than saving what the network saw. CRF 18, tuned for Doom's flat textures, keeps the files small enough for GitHub. Audio capture is silent on my Mac, so locally generated videos have no sound. Encoding takes a while because the encoder is configured to prioritize compression quality over speed.

In [ ]:
SAMPLE_RATE = 22050
VIDEO_PARAMS = dict(  # x264 at constant quality: CRF 18 is visually lossless, and "animation" suits Doom's flat textures
    fps=35, codec="libx264", quality=None, macro_block_size=8,
    ffmpeg_params=["-crf", "18", "-preset", "veryslow", "-tune", "animation", "-movflags", "+faststart"],
)


def lmp_to_mp4(lmp_path):
    """Replay an .lmp at 720p with HUD and audio, and write it next to the .lmp as .mp4."""
    lmp_path = Path(lmp_path)
    mp4_path = lmp_path.with_suffix(".mp4")
    silent_path = lmp_path.with_name(lmp_path.stem + "_silent.mp4")
    wav_path = lmp_path.with_suffix(".wav")

    game = vzd.DoomGame()
    game.load_config(scenario_cfg)
    if DOOM_WAD:
        game.set_doom_game_path(DOOM_WAD)
    game.set_screen_resolution(vzd.ScreenResolution.RES_1280X720)
    game.set_render_hud(True)
    game.set_render_corpses(True)
    game.set_window_visible(False)
    game.set_audio_buffer_enabled(True)
    game.set_audio_sampling_rate(vzd.SamplingRate.SR_22050)
    game.set_audio_buffer_size(1)
    game.add_game_args("+snd_efx 0")
    game.set_mode(vzd.Mode.SPECTATOR)
    game.init()

    short_lmp = Path("_replay_demo.lmp")  # short and relative, for the path length limit
    shutil.copy2(lmp_path, short_lmp)
    frames, audio = [], []
    try:
        game.replay_episode(str(short_lmp))
        while not game.is_episode_finished():
            state = game.get_state()
            if state is not None:
                frames.append(np.transpose(state.screen_buffer, (1, 2, 0)))
                if state.audio_buffer is not None:
                    audio.append(state.audio_buffer)
            game.advance_action()
    finally:
        game.close()
        short_lmp.unlink()

    imageio.mimsave(silent_path, frames, **VIDEO_PARAMS)

    audio = np.concatenate(audio) if audio else np.zeros(1)
    if not audio.any():  # no sound captured (e.g. on macOS): keep the silent video
        silent_path.replace(mp4_path)
        return mp4_path

    wavfile.write(wav_path, SAMPLE_RATE, audio)
    subprocess.run(
        [imageio_ffmpeg.get_ffmpeg_exe(), "-y", "-i", silent_path, "-i", wav_path,
         "-c:v", "copy", "-c:a", "aac", "-shortest", mp4_path],
        check=True, capture_output=True,
    )
    silent_path.unlink()
    wav_path.unlink()
    return mp4_path

In [ ]:
mp4s = [lmp_to_mp4(lmp) for lmp in lmps]
mp4s

For a quick preview, the highest-scoring episode is converted to a deliberately small GIF that plays directly in the notebook and on GitHub. The full-resolution MP4s remain in `artifacts/video`.

In [ ]:
GIF_WIDTH = 240
GIF_FPS = 10


def mp4_to_gif(mp4_path):
    """Write a small GIF next to an MP4, with a palette built from the video so the colors hold up."""
    gif_path = Path(mp4_path).with_suffix(".gif")
    vf = (
        f"fps={GIF_FPS},scale={GIF_WIDTH}:-1:flags=lanczos,split[a][b];"
        "[a]palettegen=max_colors=128:stats_mode=diff[p];[b][p]paletteuse=dither=bayer:bayer_scale=5:diff_mode=rectangle"
    )
    subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(), "-y", "-i", mp4_path, "-vf", vf, gif_path], check=True, capture_output=True)
    return gif_path


best_mp4 = max(
    (m for m in mp4s if m.name.startswith("best")),
    key=lambda m: lmp_rewards[m.with_suffix(".lmp")],
)
best_gif = mp4_to_gif(best_mp4)
print(f"{best_gif.name}  reward={lmp_rewards[best_mp4.with_suffix('.lmp')]:.2f}  ({best_gif.stat().st_size / 2**20:.1f} MB)")
display(Image(filename=str(best_gif)))

# Results
A short summary of the run, also saved to `artifacts/results.json` so runs are easy to compare later.

In [ ]:
results = {
    "run": run_dir.name,
    "scenario": SCENARIO_NAME,
    "iwad": IWAD_NAME,
    "best_checkpoint": best_checkpoint.name,
    "best_generation": best_gen,
    "best_steps": int(metrics.loc[best_gen, "steps"]),
    "generations": len(history),
    "eval_mean": round(float(metrics.loc[best_gen, "eval_mean"]), 4),
    "eval_std": round(float(metrics.loc[best_gen, "eval_std"]), 4),
    "eval_rewards": [round(float(r), 4) for r in eval_rewards[best_gen]],
    "best_video": [str(m.relative_to(experiment_dir)) for m in mp4s if m.name.startswith("best")],
    "best_gif": str(best_gif.relative_to(experiment_dir)),
    "net_config": NET_CONFIG,
    "shaping_hp": SHAPING_HP,
    "min_step": MIN_STEP,
    "ppo_hp": PPO_HP,
    "icm_hp": ICM_HP,
}
results_path.write_text(json.dumps(results, indent=2))
pd.Series(results, name="value").to_frame()

# Wrapping Up
My Way Home is where plain exploration struggles. Doomguy has to recognize the rooms and walk to the vest from a random start, and the only reward shows up when he gets there. Curiosity gives him something to chase in the meantime: every unfamiliar room pays out until it isn't unfamiliar anymore.

The interesting comparison is against the same notebook with `INTRINSIC_COEF = 0`, which is plain PPO with everything else held fixed. If the curiosity run finds the vest and the baseline doesn't, ICM did its job. The ICM charts are the first place to look if it doesn't: an inverse accuracy stuck at chance means the feature embedding never learned anything useful to be curious about.

Since every piece is its own small class or function, this is also a good base for trying other exploration bonuses. Swapping `ICM` and `forward_errors` for something like Random Network Distillation leaves the rest of the PPO loop untouched.

Happy Coding!